# Cryptographic Faithful Aggregation on Edge-IIoTset (local run)

**What this does:** runs the federated oilfield-IDS experiments on the real Edge-IIoTset data on your machine and writes one small `results_pack.zip` (aggregate numbers + plots, **no raw data**) that you upload back to the chat.

**Before you start**
1. Put these two files in the same folder as this notebook: `faithful_agg.py` (the core library) and `DNN-EdgeIIoT-dataset.csv` (full file is fine, it is read in chunks).
2. `pip install numpy pandas matplotlib` (optional: `tqdm` for a progress bar).
3. Run first with `PROFILE = "tune"` (FedAvg only, no attacks): the printed balanced accuracy must be well above 0.5 (aim for 0.9+) before the sweep means anything. Then `"quick"`, then `"full"`.
4. The run saves after every experiment. If it stops, re-run the cells and it resumes where it left off (delete the results folder if you change the config).

**Sections:** 1 config - 2 data - 3 model/attacks/aggregators - 4 experiment loop - 5 summary and plots - 6 cost benchmark (real commitments) - 7 export.

In [ ]:
# ============================== 1. CONFIG ==============================
import os, sys, json, time, math, platform, zipfile, datetime, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

DATA_PATH = os.environ.get("EDGE_DATA", "DNN-EdgeIIoT-dataset.csv")   # full CSV or your sample
CORE_DIR  = os.environ.get("CORE_DIR", ".")                           # folder containing faithful_agg.py
PROFILE   = os.environ.get("PROFILE", "quick")                        # "quick" or "full"
OUT_DIR   = Path(os.environ.get("OUT_DIR", "results_" + PROFILE))

COMMON = dict(
    n_sites=10, n_byz=2, dirichlet_alpha=0.5,      # 10 sites, 2 Byzantine (20%), non-IID strength
    hidden=16, max_features=40,                    # MLP width, number of input features kept
    lr=0.1, batch=128, local_steps=30,             # local SGD per round
    n_normal=100_000, n_per_attack=10_000,         # rows sampled from the CSV (None = keep all)
    root_size=1500, test_frac=0.2,                 # clean server-side root set, global test split
    krum_f=2, trim_k=2,                            # defender's ASSUMED number of attackers (fixed across scenarios)
    kappa=None,                                    # public norm bound B_t = kappa * ||root update||; None = auto-calibrate
    arm_filter=None,                               # optional list of arm names to run (used by the "tune" profile)
)
PROFILES = {
    "quick": dict(seeds=[0, 1], rounds=15, taus=[0.0, 0.2],
                  scenarios=["none", "signflip", "labelflip", "adaptive"],
                  n_normal=20_000, n_per_attack=2_000),
    "full":  dict(seeds=[0, 1, 2, 3, 4], rounds=40, taus=[0.0, 0.1, 0.2, 0.3],
                  scenarios=["none", "signflip", "scale", "noise", "labelflip", "adaptive"]),
    # "tune": check that the IDS actually learns BEFORE the big sweep (FedAvg + root-reference Faithful, no attack, 1 seed)
    "tune":  dict(seeds=[0], rounds=40, taus=[0.0], scenarios=["none"],
                  arm_filter=["fedavg", "faithful_root_t0.0", "local_only"]),
}
CFG = {**COMMON, **PROFILES[PROFILE]}
CFG.update(json.loads(os.environ.get("CFG_OVERRIDES", "{}")))       # optional JSON overrides

sys.path.insert(0, str(Path(CORE_DIR).resolve()))
import faithful_agg as fa
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "figs").mkdir(exist_ok=True)
json.dump(CFG, open(OUT_DIR / "config.json", "w"), indent=2)
print("faithful_agg:", fa.__file__)
print("profile:", PROFILE, "| output folder:", OUT_DIR.resolve())
print(json.dumps(CFG, indent=1))

## 2. Data: chunked sampling, preprocessing, feature selection
The CSV is scanned twice in chunks (labels first, then a per-class Bernoulli sample), so memory stays small. Identifier/payload columns (IPs, timestamps, ports, raw payloads) are dropped to avoid shortcut learning; low-cardinality text columns are one-hot encoded; the top `max_features` features by absolute correlation with the binary label are kept. Task: **binary IDS** (normal vs attack, column `Attack_label`).

In [ ]:
# ============================== 2. DATA ==============================
DROP_SET = {"frame.time", "ip.src_host", "ip.dst_host", "arp.src.proto_ipv4", "arp.dst.proto_ipv4",
            "http.file_data", "http.request.full_uri", "icmp.transmit_timestamp", "http.request.uri.query",
            "tcp.options", "tcp.payload", "tcp.srcport", "tcp.dstport", "udp.port", "mqtt.msg"}

def find_col(cols, name):
    for c in cols:
        if c.strip().lower() == name.lower():
            return c
    return None

head = pd.read_csv(DATA_PATH, nrows=5, low_memory=False)
YCOL, TCOL = find_col(head.columns, "Attack_label"), find_col(head.columns, "Attack_type")
assert YCOL and TCOL, f"label columns not found; last columns are {list(head.columns[-4:])}"

lab = pd.read_csv(DATA_PATH, usecols=[YCOL, TCOL], low_memory=False)
lab[TCOL] = lab[TCOL].astype(str)
counts = lab.groupby(TCOL)[YCOL].agg(["size", "mean"])
normal_types = set(counts.index[counts["mean"] == 0])
cap = {t: (CFG["n_normal"] if t in normal_types else CFG["n_per_attack"]) for t in counts.index}
keep_p = {t: (1.0 if cap[t] is None else min(1.0, cap[t] / counts.loc[t, "size"])) for t in counts.index}
print(f"{len(lab):,} rows in file; classes:\n{counts}")
FILE_ROWS, FILE_COUNTS = int(len(lab)), counts["size"].astype(int).to_dict()
del lab

rng_s = np.random.default_rng(12345)
parts = []
for ch in pd.read_csv(DATA_PATH, chunksize=250_000, low_memory=False):
    p = ch[TCOL].astype(str).map(keep_p).fillna(0).values
    parts.append(ch[rng_s.random(len(ch)) < p])
raw = pd.concat(parts, ignore_index=True)
del parts

y_arr = raw[YCOL].astype(int).values
types_arr = raw[TCOL].astype(str).values
X = raw.drop(columns=[YCOL, TCOL])
X = X.drop(columns=[c for c in X.columns if c.strip() in DROP_SET], errors="ignore")
cat_cols = []
for c in X.columns:
    if X[c].dtype == object:
        conv = pd.to_numeric(X[c], errors="coerce")
        if conv.notna().mean() > 0.9:
            X[c] = conv
        else:
            cat_cols.append(c)
dummies = [pd.get_dummies(X[c].astype(str), prefix=c, dtype=np.float32) for c in cat_cols if X[c].nunique() <= 20]
X = X.drop(columns=cat_cols)
if dummies:
    X = pd.concat([X] + dummies, axis=1)
X = X.apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(np.float32)
X = X.loc[:, X.std() > 0]
corr = X.corrwith(pd.Series(y_arr, index=X.index)).abs().fillna(0.0).sort_values(ascending=False)
FEATURES = list(corr.index[: CFG["max_features"]])
X_all = X[FEATURES].values.astype(np.float32)
del raw, X
print(f"\nsampled rows: {len(y_arr):,} | attack share: {y_arr.mean():.2%} | features kept: {len(FEATURES)}")
print(pd.Series(types_arr).value_counts().to_string())

## 3. Model, attacks, baseline aggregators
* **IDS model:** one-hidden-layer MLP in NumPy (flattened to a single parameter vector, so it plugs straight into the secure protocol).
* **Public bound:** `B_t = kappa * ||root update||` each round (the root set is a small clean server-side capture, e.g. testbed traffic).
* **Baselines:** FedAvg, coordinate-wise trimmed mean, multi-Krum (both told a fixed assumed attacker count), secure norm-bound only, and Faithful Aggregation with the previous-aggregate reference or the clean root-set reference, over a sweep of the cosine threshold tau.
* **Attacks (2 of 10 sites):** `signflip` (-4u), `scale` (25u), `noise`, `labelflip` (data poisoning, honest-looking norm), `adaptive` (worst case: knows B, tau and the reference; submits norm 0.95B at the minimum allowed cosine with the rest along a fixed orthogonal drift).

In [ ]:
# ============================== 3. MODEL / ATTACKS / AGGREGATORS ==============================
F = X_all.shape[1]; H = CFG["hidden"]; D = F * H + H + H + 1
print("model dimension d =", D)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def unpack(w):
    i = 0
    W1 = w[i:i + F * H].reshape(F, H); i += F * H
    b1 = w[i:i + H]; i += H
    W2 = w[i:i + H].reshape(H, 1); i += H
    return W1, b1, W2, w[i:i + 1]

def init_w(rng):
    W1 = rng.normal(0, np.sqrt(2 / F), (F, H)); W2 = rng.normal(0, np.sqrt(2 / H), (H, 1))
    return np.concatenate([W1.ravel(), np.zeros(H), W2.ravel(), np.zeros(1)])

def predict(w, Xm):
    W1, b1, W2, b2 = unpack(w)
    return sigmoid(np.maximum(Xm @ W1 + b1, 0.0) @ W2 + b2).ravel()

def local_update(w, Xm, ym, rng):
    w0 = w.copy(); w = w.copy(); n = len(ym)
    for _ in range(CFG["local_steps"]):
        idx = rng.integers(0, n, CFG["batch"])
        xb = Xm[idx].astype(np.float64); yb = ym[idx].astype(np.float64)
        W1, b1, W2, b2 = unpack(w)
        z1 = xb @ W1 + b1; a1 = np.maximum(z1, 0.0)
        p = sigmoid(a1 @ W2 + b2).ravel()
        dz2 = ((p - yb) / len(yb))[:, None]
        da1 = dz2 @ W2.T; dz1 = da1 * (z1 > 0)
        g = np.concatenate([(xb.T @ dz1).ravel(), dz1.sum(0), (a1.T @ dz2).ravel(), dz2.sum(0)])
        w -= CFG["lr"] * g
    return w - w0

def metrics(w, Xm, ym, auc=False):
    sc = predict(w, Xm); pr = sc > 0.5; yb = ym.astype(bool)
    tp = float((pr & yb).sum()); fp = float((pr & ~yb).sum()); fn = float((~pr & yb).sum()); tn = float((~pr & ~yb).sum())
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    out = dict(acc=(tp + tn) / len(ym), f1=2 * prec * rec / max(prec + rec, 1e-12), recall=rec,
               fpr=fp / max(fp + tn, 1), bal_acc=0.5 * (rec + tn / max(tn + fp, 1)))   # bal_acc: 0.5 = chance
    if auc:
        rk = pd.Series(sc).rank().values; npos = float(yb.sum()); nneg = len(yb) - npos
        out["auroc"] = float((rk[yb].sum() - npos * (npos + 1) / 2) / max(npos * nneg, 1))
    return out

def agg_trimmed(U, k):
    if k > 0 and len(U) > 2 * k:
        return np.sort(U, axis=0)[k:len(U) - k].mean(0)
    return U.mean(0)

def agg_multikrum(U, f):
    n = len(U); G = U @ U.T; sq = np.diag(G); D2 = sq[:, None] + sq[None, :] - 2 * G
    m_close = max(1, n - f - 2)
    scores = [np.sort(np.delete(D2[i], i))[:m_close].sum() for i in range(n)]
    sel = np.argsort(scores)[:max(1, n - f)]
    return U[sel].mean(0), set(int(s) for s in sel)

def adaptive_vec(B, ref, tau, drift):
    if ref is None or np.linalg.norm(ref) == 0:
        return 0.95 * B * drift
    rh = ref / np.linalg.norm(ref)
    p = drift - (drift @ rh) * rh; p /= np.linalg.norm(p)
    c = min(max(tau or 0.0, 0.0) + 0.02, 0.999)
    return 0.95 * B * (c * rh + np.sqrt(1 - c * c) * p)

def dirichlet_partition(train_idx, rng, tries=300, min_size=300):
    classes = np.unique(types_arr[train_idx])
    for _ in range(tries):
        parts = [[] for _ in range(CFG["n_sites"])]
        for t in classes:
            idx = rng.permutation(train_idx[types_arr[train_idx] == t])
            pr = rng.dirichlet([CFG["dirichlet_alpha"]] * CFG["n_sites"])
            cuts = (np.cumsum(pr) * len(idx)).astype(int)[:-1]
            for s, chunk in enumerate(np.split(idx, cuts)):
                parts[s].extend(chunk.tolist())
        if all(len(p) >= min_size for p in parts):
            fr = [y_arr[p].mean() for p in parts]
            if all(0.05 <= f <= 0.95 for f in fr):
                return [np.array(p) for p in parts]
    raise RuntimeError("partition constraints not met: raise dirichlet_alpha or lower min_size / use more rows")

def build_bundle(seed):
    rng = np.random.default_rng(1000 + seed)
    test, root, train = [], [], []
    for t in np.unique(types_arr):
        idx = rng.permutation(np.where(types_arr == t)[0])
        nt = max(1, int(round(len(idx) * CFG["test_frac"]))); test.append(idx[:nt]); rest = idx[nt:]
        nr = max(1, int(round(CFG["root_size"] * len(idx) / len(types_arr)))); root.append(rest[:nr]); train.append(rest[nr:])
    test, root, train = (np.concatenate(a) for a in (test, root, train))
    mu = X_all[train].mean(0); sd = X_all[train].std(0) + 1e-6
    Z = np.clip((X_all - mu) / sd, -5, 5).astype(np.float32)
    parts = dirichlet_partition(train, rng)
    return dict(sites=[(Z[p], y_arr[p]) for p in parts], root=(Z[root], y_arr[root]), test=(Z[test], y_arr[test]),
                site_sizes=[len(p) for p in parts], site_attack_frac=[float(y_arr[p].mean()) for p in parts])

In [ ]:
# ---- auto-calibrate kappa (public norm bound multiplier) on CLEAN rounds, seed 0 ----
if CFG["kappa"] is None:
    b0 = build_bundle(0); rng = np.random.default_rng(99); w = init_w(rng); ratios = []
    for r in range(6):
        ru = local_update(w, *b0["root"], rng); rn = np.linalg.norm(ru)
        ups = [local_update(w, Xs, ys, rng) for Xs, ys in b0["sites"]]
        ratios += [np.linalg.norm(u) / rn for u in ups]
        w = w + np.mean(ups, axis=0)
    p50, p99, pmax = np.percentile(ratios, [50, 99, 100])
    KAPPA = max(2.0, math.ceil(1.25 * p99 * 2) / 2)
    print(f"honest ||u||/||root|| : median {p50:.2f}, p99 {p99:.2f}, max {pmax:.2f}  ->  kappa = {KAPPA}")
    del b0
else:
    KAPPA = float(CFG["kappa"])
CFG["kappa_used"] = KAPPA
json.dump(CFG, open(OUT_DIR / "config.json", "w"), indent=2)

## 4. Experiment loop (resumable)
Each run = (arm, scenario, seed). Results are appended to `results.csv` after every run. Secure arms run the real secret-sharing + Beaver-triple tests; commitments/Merkle are switched off here (they do not change accept decisions or the aggregate) and are measured separately in section 6.

In [ ]:
# ============================== 4. EXPERIMENT LOOP ==============================
ARMS = [("fedavg", "fedavg", None), ("trimmed_mean", "trimmed", None), ("multi_krum", "krum", None),
        ("secure_norm_only", "norm_only", None)]
ARMS += [(f"faithful_prev_t{t}", "faithful_prev", t) for t in CFG["taus"]]
ARMS += [(f"faithful_root_t{t}", "faithful_root", t) for t in CFG["taus"]]
if CFG["arm_filter"]:
    ARMS = [a for a in ARMS if a[0] in CFG["arm_filter"]]
RES_PATH = OUT_DIR / "results.csv"

def run_one(arm, fam, tau, scn, seed, bundle):
    t_start = time.time()
    rng = np.random.default_rng(10_000 + seed); crng = np.random.default_rng(20_000 + seed)
    sites, (Xr, yr), (Xt, yt) = bundle["sites"], bundle["root"], bundle["test"]
    n = len(sites); n_byz = 0 if scn == "none" else CFG["n_byz"]; byz = set(range(n_byz))
    w = init_w(rng); prev = None
    drift = rng.normal(size=D); drift /= np.linalg.norm(drift)
    hr = hb = ht = bt = 0; curve = []
    for r in range(CFG["rounds"]):
        root_u = local_update(w, Xr, yr, rng); B = KAPPA * np.linalg.norm(root_u)
        ref = root_u if fam == "faithful_root" else (prev if fam == "faithful_prev" else None)
        ups = []
        for k, (Xs, ys) in enumerate(sites):
            u = local_update(w, Xs, (1 - ys) if (k in byz and scn == "labelflip") else ys, rng)
            if k in byz:
                if scn == "signflip": u = -4.0 * u
                elif scn == "scale": u = 25.0 * u
                elif scn == "noise": u = rng.normal(size=D) * 2.0
                elif scn == "adaptive": u = adaptive_vec(B, ref, tau, drift)
            ups.append(u)
        U = np.array(ups); mask = None
        if fam == "fedavg":
            agg = U.mean(0)
        elif fam == "trimmed":
            agg = agg_trimmed(U, CFG["trim_k"])
        elif fam == "krum":
            agg, sel = agg_multikrum(U, CFG["krum_f"]); mask = [i in sel for i in range(n)]
        else:
            subs = [fa.make_submission(k, u, crng, with_commitments=False) for k, u in enumerate(ups)]
            res = fa.run_round(subs, crng, B, tau, ref, verify_openings=False)
            agg = res.mean_update; mask = [bool(b) for b in res.bits]
            if res.n_accepted: prev = res.mean_update
        if mask is not None:
            for k in range(n):
                if k in byz: bt += 1; hb += (not mask[k])
                else: ht += 1; hr += (not mask[k])
        w = w + agg
        curve.append(round(metrics(w, Xt, yt)["bal_acc"], 4))
    m = metrics(w, Xt, yt, auc=True)
    return dict(arm=arm, family=fam, tau=tau, scenario=scn, seed=seed, n_byz=n_byz, rounds=CFG["rounds"],
                final_acc=m["acc"], final_f1=m["f1"], final_bal_acc=m["bal_acc"], final_auroc=m["auroc"],
                final_recall=m["recall"], final_fpr=m["fpr"], last3_bal=float(np.mean(curve[-3:])),
                honest_rejected=hr if mask is not None else np.nan,
                honest_total=ht if mask is not None else np.nan, byz_rejected=hb if mask is not None else np.nan,
                byz_total=bt if mask is not None else np.nan, curve_bal_acc=json.dumps(curve), seconds=round(time.time() - t_start, 1))

def run_local_only(seed, bundle):
    rng = np.random.default_rng(30_000 + seed); Xt, yt = bundle["test"]; ms = []
    for Xs, ys in bundle["sites"]:
        w = init_w(rng)
        for r in range(CFG["rounds"]):
            w = w + local_update(w, Xs, ys, rng)
        ms.append(metrics(w, Xt, yt, auc=True))
    g = lambda k: float(np.mean([m[k] for m in ms]))
    return dict(arm="local_only", family="local_only", tau=None, scenario="none", seed=seed, n_byz=0, rounds=CFG["rounds"],
                final_acc=g("acc"), final_f1=g("f1"), final_bal_acc=g("bal_acc"), final_auroc=g("auroc"),
                final_recall=g("recall"), final_fpr=g("fpr"), last3_bal=g("bal_acc"),
                honest_rejected=np.nan, honest_total=np.nan, byz_rejected=np.nan, byz_total=np.nan, curve_bal_acc="[]", seconds=0)

done = set()
if RES_PATH.exists():
    old = pd.read_csv(RES_PATH); done = set(zip(old.arm, old.scenario, old.seed))
    print(f"resuming: {len(done)} runs already done")
total = len(CFG["seeds"]) * (len(ARMS) * len(CFG["scenarios"]) + 1); count = 0; t0 = time.time()
bundle_info = {}
for seed in CFG["seeds"]:
    bundle = build_bundle(seed)
    bundle_info[seed] = dict(site_sizes=bundle["site_sizes"], site_attack_frac=bundle["site_attack_frac"],
                             n_test=len(bundle["test"][1]), n_root=len(bundle["root"][1]))
    jobs = ([("local_only", "local_only", None, "none")] if (not CFG["arm_filter"] or "local_only" in CFG["arm_filter"]) else []) \
           + [(a, f, t, s) for (a, f, t) in ARMS for s in CFG["scenarios"]]
    for arm, fam, tau, scn in jobs:
        count += 1
        if (arm, scn, seed) in done:
            continue
        row = run_local_only(seed, bundle) if fam == "local_only" else run_one(arm, fam, tau, scn, seed, bundle)
        pd.DataFrame([row]).to_csv(RES_PATH, mode="a", header=not RES_PATH.exists(), index=False)
        print(f"[{count}/{total}] seed={seed} {scn:10s} {arm:22s} balAcc={row['final_bal_acc']:.3f} AUROC={row['final_auroc']:.3f}  ({time.time() - t0:.0f}s elapsed)")
    del bundle
print("done.")

## 5. Summary tables and plots

In [ ]:
# ============================== 5. SUMMARY & PLOTS ==============================
import matplotlib.pyplot as plt
res = pd.read_csv(RES_PATH)
res["honest_rej_rate"] = res.honest_rejected / res.honest_total.replace(0, np.nan)
res["byz_rej_rate"] = res.byz_rejected / res.byz_total.replace(0, np.nan)
ci = lambda x: 1.96 * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan
summary = res.groupby(["scenario", "arm"]).agg(n=("seed", "count"), bal_acc=("final_bal_acc", "mean"), bal_ci95=("final_bal_acc", ci),
        auroc=("final_auroc", "mean"), f1=("final_f1", "mean"),
        recall=("final_recall", "mean"), fpr=("final_fpr", "mean"), acc=("final_acc", "mean"),
        honest_rej=("honest_rej_rate", "mean"), byz_rej=("byz_rej_rate", "mean")).reset_index()
summary.to_csv(OUT_DIR / "summary.csv", index=False)
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 500)
print(summary.round(3).to_string(index=False))

order = [a for a, _, _ in ARMS] + ["local_only"]
scns = [s for s in CFG["scenarios"]]
fig, axes = plt.subplots(1, len(scns), figsize=(4.2 * len(scns), 5), sharex=True)
axes = np.atleast_1d(axes)
for ax, scn in zip(axes, scns):
    sub = summary[summary.scenario == scn].set_index("arm").reindex([a for a in order if a in set(summary[summary.scenario == scn].arm)])
    ax.barh(sub.index, sub.bal_acc, xerr=sub.bal_ci95.fillna(0), color="#4C78A8"); ax.set_title(scn); ax.set_xlim(0.4, 1)
    ax.set_xlabel("final balanced accuracy (0.5 = chance)")
plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / "f1_by_arm.png", dpi=130); plt.show()

fam = res[res.family.isin(["faithful_prev", "faithful_root"]) & (res.scenario != "none")]
if len(fam):
    g = fam.groupby(["scenario", "family", "tau"]).agg(ba=("final_bal_acc", "mean"), hrej=("honest_rej_rate", "mean"), brej=("byz_rej_rate", "mean")).reset_index()
    sc2 = [s for s in scns if s != "none"]
    fig, axes = plt.subplots(2, len(sc2), figsize=(4.2 * len(sc2), 6), sharex=True, squeeze=False)
    for j, scn in enumerate(sc2):
        for famn, col in (("faithful_prev", "#E45756"), ("faithful_root", "#54A24B")):
            s = g[(g.scenario == scn) & (g.family == famn)].sort_values("tau")
            axes[0][j].plot(s.tau, s.ba, "o-", color=col, label=famn); axes[1][j].plot(s.tau, s.hrej, "o-", color=col)
        axes[0][j].set_title(scn); axes[0][j].set_ylim(0.4, 1); axes[1][j].set_xlabel("cosine threshold tau")
    axes[0][0].set_ylabel("final balanced accuracy"); axes[1][0].set_ylabel("honest updates rejected"); axes[0][0].legend()
    plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / "tau_tradeoff.png", dpi=130); plt.show()

## 6. Cost benchmark with REAL commitments, Merkle inclusion and audit
One round at the real model dimension. Pedersen commitments here use a toy 512-bit group (cost scaling only); communication is reported for 256-bit group elements. Secure comparison is an ideal functionality, so its bytes are a placeholder and are reported separately.

In [ ]:
# ============================== 6. COST BENCHMARK ==============================
b0 = build_bundle(0); rb = np.random.default_rng(6); rc = np.random.default_rng(5)
w0 = init_w(rb); ups = [local_update(w0, Xs, ys, rb) for Xs, ys in b0["sites"]]
root_u = local_update(w0, *b0["root"], rb); Bq = KAPPA * np.linalg.norm(root_u)
t = time.perf_counter(); subs = [fa.make_submission(k, u, rc, with_commitments=True) for k, u in enumerate(ups)]; t_client = time.perf_counter() - t
led = fa.Ledger(); t = time.perf_counter()
res_c = fa.run_round(subs, rc, Bq, tau=0.2, ref=root_u, verify_openings=True, ledger=led); t_round = time.perf_counter() - t
t = time.perf_counter(); audit_ok = fa.audit(res_c.transcript); t_audit = time.perf_counter() - t
incl_ok = all(fa.site_verify(res_c.transcript, i, j)[0] for i in range(len(subs)) for j in range(2))
n = len(subs); placeholder_cmp = n * 3 * 2 * fa.CMP_ELEMS * fa.FE_BYTES
plain_bytes = n * D * 4
cost = dict(model_dim=D, n_sites=n, audit_ok=bool(audit_ok), inclusion_proofs_ok=bool(incl_ok),
            accepted=int(res_c.n_accepted), bytes_by_phase={k: int(v) for k, v in led.bytes_.items()},
            rounds_by_phase={k: int(v) for k, v in led.rounds.items()}, secs_by_phase={k: round(v, 3) for k, v in led.secs.items()},
            client_commit_and_share_secs_total=round(t_client, 3), server_round_secs=round(t_round, 3), audit_secs=round(t_audit, 3),
            total_online_bytes=int(led.total_bytes()), placeholder_comparison_bytes=int(placeholder_cmp),
            online_bytes_excluding_placeholder=int(led.total_bytes() - placeholder_cmp),
            plain_fedavg_upload_bytes_float32=int(plain_bytes),
            ratio_vs_plain=round(led.total_bytes() / plain_bytes, 2),
            ratio_vs_plain_excluding_placeholder=round((led.total_bytes() - placeholder_cmp) / plain_bytes, 2),
            offline_triple_bytes=int(led.bytes_.get("offline", 0)))
json.dump(cost, open(OUT_DIR / "cost.json", "w"), indent=2)
print(json.dumps(cost, indent=1))

## 7. Export the results pack (aggregates only, no raw data)

In [ ]:
# ============================== 7. EXPORT ==============================
audit_info = dict(file_rows=FILE_ROWS, file_class_counts=FILE_COUNTS, sampled_rows=int(len(y_arr)),
                  sampled_attack_share=float(y_arr.mean()), sampled_class_counts=pd.Series(types_arr).value_counts().to_dict(),
                  n_features=len(FEATURES), feature_names=FEATURES, model_dim=D, kappa=KAPPA, partitions=bundle_info)
json.dump(audit_info, open(OUT_DIR / "data_audit.json", "w"), indent=1)
env = dict(python=sys.version, platform=platform.platform(), numpy=np.__version__, pandas=pd.__version__,
           cpu_count=os.cpu_count(), finished=datetime.datetime.now().isoformat(timespec="seconds"))
json.dump(env, open(OUT_DIR / "env.json", "w"), indent=1)
zip_path = OUT_DIR / "results_pack.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["config.json", "results.csv", "summary.csv", "cost.json", "data_audit.json", "env.json"]:
        z.write(OUT_DIR / name, name)
    for p in (OUT_DIR / "figs").glob("*.png"):
        z.write(p, f"figs/{p.name}")
print(f"wrote {zip_path.resolve()}  ({zip_path.stat().st_size / 1024:.0f} KiB)")
print("Upload results_pack.zip to the chat. It contains only aggregate metrics, plots and settings.")